In [ ]:
# track

findex = 26 #145 #23 #123 #111 # 23, 66
sd_findex = 23
species = 5

In [ ]:
%load_ext autoreload
%autoreload 2

from graphet import Data
from graphet.plugins import TristanV2 
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import sys
sys.path.append('../v2_proc_files')
from useful_funcs import *
from TristanRun import TristanRun
from FieldLineHelper import FieldLineHelper 
from ShellData import ShellData

plt.close('all')

%matplotlib widget

set_standard_plot_params()

# Read Input File Data

run = TristanRun(findex)
gyrofreq = run.gyrofreq
boilerplate = run.boilerplate + f" tstep = {run.steps[0]}"
cc = run.cc
B0 = run.B0

my_steps = run.steps[:15]
my_steps = [run.steps[-1]]
my_steps = run.steps
nsteps = len(my_steps)

In [ ]:
# Configure Data

d = run.config_data(my_steps, "fields")

In [ ]:
# Download Shell Data
sd_run = TristanRun(sd_findex)

sd = ShellData(sd_run, species, sd_run.steps[:-1])
sd.calc_quants()
p = sd.p
code = sd.code

sd.print_summary()

In [ ]:
# def decimal_index(A, coord):
#     xlen, ylen = A.shape
#     return A[int(coord[0] * xlen), int(coord[1] * ylen)]

# class FieldLineHelper:
#     def __init__(self, run, N, contour_ds = 1, origin = [0.0, 0.0]):
#         self.RUN = run
#         self.N = N
#         self.levels = np.full(N, np.inf)
#         self.gauge_integral = 0
#         self.ADJ = 0.05
#         self.contour_ds = contour_ds
#         self.lspace = 0

#         self.origin = origin

#         if not (0 <= self.origin[0] <= 1) or not (0 <= self.origin[1] <= 1):
#             raise ValueError("Origin coordinates must be between 0 and 1")

#     def calculate_levels(self, az_min, az_max):
#         self.levels = [level for level in self.levels if level <= az_max]

#         if (self.lspace == 0 or len(self.levels) == 0):
#             if (self.lspace != 0 and len(self.levels) == 0):
#                 print("All levels gone")

#             left = az_min + self.ADJ * (az_max - az_min)
#             right = az_max - self.ADJ * (az_max - az_min)

#             self.levels = np.linspace(left, right, self.N)
#             self.lspace = self.levels[1] - self.levels[0]

#         while (self.levels[0] - self.lspace > az_min):
#             self.levels = np.insert(self.levels, 0, self.levels[0] - self.lspace)

#     def plot_field_lines(self, roi, Ez_representative, plot = plt, arrow_spacing = 150):
#         ds_roi = roi.coarsen(x=self.contour_ds, y=self.contour_ds, boundary='trim').mean()

#         Azp = self.compute_Az(ds_roi) + self.gauge_integral

#         self.calculate_levels(np.min(Azp), np.max(Azp))

#         contours = plot.contour(Azp.x, Azp.y, Azp, levels = np.sort(self.levels), colors = 'white', linestyles = 'solid', linewidths = 0.4)

#         if (arrow_spacing != -1):
#             for collection in contours.collections:
#                 for path in collection.get_paths():
#                     verts = path.vertices
#                     if len(verts) > 1:  # Only add arrows for lines with multiple points
#                         # Add an arrow marker at every nth point to avoid overcrowding
#                         for i in range(0, len(verts) - 1, arrow_spacing):
#                             dx = verts[i+1, 0] - verts[i, 0]
#                             dy = verts[i+1, 1] - verts[i, 1]
#                             plot.annotate('', xy=verts[i+1], xytext=verts[i],
#                                         arrowprops=dict(arrowstyle='->', color='white', lw=0.8))   

#         self.gauge_integral -= self.RUN.cc * self.RUN.output_interval * Ez_representative

#     def compute_Az(self, ds_roi):    
#         Az1 = np.cumsum(ds_roi.bx, axis=0) - np.cumsum(ds_roi.by, axis=1)
#         Az2 = decimal_index(Az1, self.origin)

#         return (Az1 - Az2) * self.contour_ds * self.RUN.istep

In [ ]:
print(run.steps)

In [ ]:
sel_prtl = 28052

box = [5, run.xlim - 5, 0, run.ylim]
tindex = np.where(~np.isnan(sd.gammas[:, sel_prtl]))[0][-1]

flh = FieldLineHelper(run, 8, contour_ds = 5, origin = [0, 0]) #20

plt.close()

roi = fq_cut(d.fields, tindex // 5, box)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize = (12, 5))
fig.subplots_adjust(wspace=0)
#fig.suptitle(boilerplate + f"\n Particle \#{sel_prtl}")

# Fields Plot

ax1.set_xlim(box[:2])
ax1.set_ylim(box[2:])
ax1.set_xlabel('$x$ [$c/\omega_{p}$]')
ax1.set_ylabel('$y$ [$c/\omega_{p}$]')

ax1.plot(p[:tindex, sel_prtl, code['x']], p[:tindex, sel_prtl, code['y']], color='red', linewidth=1.5)

flh.plot_field_lines(roi, 0, arrow_spacing = -1, plot = ax1)

plot_fq3(roi, roi.dens, plot = ax1, vabsmax = 5, positive = True)

ax1.set_aspect(1)
ax1.set_title(f'$\Omega t$ = {tindex * gyrofreq:.1f}')

# Energy Plot
ax2.plot(np.arange(p.shape[0], dtype=float) * run.gyrofreq, sd.gammas[:, sel_prtl] - 1)

ax2.set_xlabel("$\Omega t$")
ax2.set_ylabel("$\gamma - 1$")
ax2.set_title(f"Particle Energy")
ax2.grid(visible = True)
ax2.set_ylim([0, np.nanmax(sd.gammas[:, sel_prtl] - 1) * 1.2])
ax2.set_xlim([0, tindex * gyrofreq])
#ax2.axvline(tindex * gyrofreq, color = 'yellow', linestyle = '--')
ax2.set_aspect('auto')

metadata = {
        "Particle": str(sel_prtl),
        "Species": str(species),
        "Field folder index": str(findex),
        "Shell folder index": str(sd_findex),
}


save_polished_fig("twostep_accel", metadata = metadata)